In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

In [30]:
df = pd.read_csv('Chronotype_NHANES_Imputation1.csv')
df.head()



,Seqn,Gender,Age,Race,BMI,Waist_C,Systolic,Diastolic,Carb_diet,HSCRP,Smokingstatus,Alcohol,Sleep_hrs,Sleeptime,Wakeuptime,Chronotype_slphrs,WakeUpCat
0,83732,1,62,2.0,27.8,101.1,122.6667,65.33334,126.0,0.6,3.0,1.0,5.5,23:30:00,05:00:00,3.0,1
1,83733,1,53,2.0,30.8,107.9,140.0000,86.00000,126.0,1.4,1.0,7.0,8.0,23:00:00,07:00:00,3.0,3
2,83734,1,78,2.0,28.8,116.5,135.3333,45.33333,96.0,0.6,3.0,0.0,7.0,22:30:00,05:30:00,2.0,2
3,83735,2,56,2.0,42.4,110.1,134.0000,70.00000,216.0,9.0,3.0,3.0,6.5,23:30:00,06:00:00,3.0,2
4,83741,1,22,3.0,28.0,86.6,111.3333,72.66666,5.5,1.3,2.0,3.0,6.5,23:00:00,05:30:00,3.0,2


### Data Preparation
We need to separate the target variable `Chronotype_slphrs` from the features. We should also remove identifiers like `Seqn` and non-numeric time columns that aren't ready for modeling.

In [31]:
# 1. Apply Winsorization (Clipping) to handle outliers
def handle_outliers(df_in, columns):
    df_out = df_in.copy()
    for col in columns:
        lower = df_out[col].quantile(0.01)
        upper = df_out[col].quantile(0.99)
        df_out[col] = df_out[col].clip(lower, upper)
    return df_out

numeric_features = ['Age', 'BMI', 'Waist_C', 'Systolic', 'Diastolic', 'Carb_diet', 'HSCRP', 'Alcohol', 'Sleep_hrs']
df_cleaned = handle_outliers(df, numeric_features)

# 2. Define features and target using clipped data, removing 'Race'
X = df_cleaned.drop(columns=['Chronotype_slphrs', 'Seqn', 'Sleeptime', 'Wakeuptime', 'Gender', 'Smokingstatus', 'Race'])
y = df_cleaned['Chronotype_slphrs']

# 3. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Outliers clipped at 1st/99th percentile for: {numeric_features}")
print(f"Training set shape: {X_train.shape}")

Outliers clipped at 1st/99th percentile for: ['Age', 'BMI', 'Waist_C', 'Systolic', 'Diastolic', 'Carb_diet', 'HSCRP', 'Alcohol', 'Sleep_hrs']
Training set shape: (4469, 10)


### Model Training and Evaluation
Now we initialize and fit the Random Forest Classifier.

In [32]:
# Adjusting hyperparameters to reduce overfitting
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=7,        # Reduced from 15
    min_samples_split=10, # Increased from 5
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)

# Calculate scores to check for improvement in overfitting gap
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"RF Training Accuracy: {train_acc:.4f}")
print(f"RF Test Accuracy: {test_acc:.4f}")

RF Training Accuracy: 0.7906
RF Test Accuracy: 0.7236


### XGBoost Classifier
Next, we'll try the XGBoost model. First, we need to import it and then train it using our prepared features and target.

In [33]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# XGBoost often requires label encoding for targets starting from 0
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Initialize the XGBoost model with regularization to handle overfitting
xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=4,           # Reduced from default to simplify trees
    learning_rate=0.05,     # Slower learning rate for better generalization
    reg_alpha=0.1,         # L1 regularization
    reg_lambda=1.0,        # L2 regularization
    random_state=42,
    eval_metric='mlogloss'
)

# Fit the model
xgb_model.fit(X_train, y_train_encoded)

# Make predictions for both sets to monitor gap
y_train_pred_xgb = xgb_model.predict(X_train)
y_test_pred_xgb = xgb_model.predict(X_test)

# Evaluation
xgb_train_acc = accuracy_score(y_train_encoded, y_train_pred_xgb)
xgb_test_acc = accuracy_score(y_test_encoded, y_test_pred_xgb)

print(f"XGBoost Training Accuracy: {xgb_train_acc:.4f}")
print(f"XGBoost Test Accuracy: {xgb_test_acc:.4f}")
print(f"Generalization Gap: {(xgb_train_acc - xgb_test_acc):.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_encoded, y_test_pred_xgb, target_names=le.classes_.astype(str)))

XGBoost Training Accuracy: 0.8308
XGBoost Test Accuracy: 0.7683
Generalization Gap: 0.0625

Classification Report (Test Set):
              precision    recall  f1-score   support

         1.0       0.75      0.49      0.59       156
         2.0       0.70      0.81      0.75       344
         3.0       0.81      0.87      0.84       443
         4.0       0.80      0.70      0.75        74
         5.0       0.86      0.63      0.73       101

    accuracy                           0.77      1118
   macro avg       0.78      0.70      0.73      1118
weighted avg       0.77      0.77      0.76      1118



### Neural Network Hyperparameter Tuning with Keras Tuner

We will now use `keras_tuner` to perform a more extensive hyperparameter search for our Neural Network model. This allows us to systematically explore different architectures and training parameters to find the best performing model. The goal is to maximize the validation accuracy.

In [34]:
# Install Keras Tuner if not already installed
import subprocess
import sys

try:
    import keras_tuner
except ImportError:
    print("Installing Keras Tuner...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "keras-tuner"])
    import keras_tuner

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

print("Keras Tuner and TensorFlow imported successfully.")

Keras Tuner and TensorFlow imported successfully.


In [35]:
# Scale features for Neural Networks and SVM
scaler = StandardScaler()
X_train_nn_scaled = scaler.fit_transform(X_train)
X_test_nn_scaled = scaler.transform(X_test)

# Get number of classes for the output layer
num_classes = len(np.unique(y_train_encoded))

# Define the Keras Tuner model building function
def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_nn_scaled.shape[1],)))

    # Tune the number of layers
    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(layers.Dense(units=hp.Int('units_' + str(i),
                                            min_value=32,
                                            max_value=512,
                                            step=32),
                                 activation='relu'))
        model.add(layers.Dropout(hp.Float('dropout_' + str(i),
                                           min_value=0.0,
                                           max_value=0.5,
                                           step=0.1)))

    # Output layer
    model.add(layers.Dense(num_classes, activation='softmax'))

    # Tune the learning rate for the optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Initialize the RandomSearch tuner
tuner = keras_tuner.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='keras_tuner_dir',
    project_name='chronotype_mlp_tuning',
    overwrite=False) # Keep results if exists

# Perform the search
tuner.search(X_train_nn_scaled, y_train_encoded,
             epochs=20,
             validation_data=(X_test_nn_scaled, y_test_encoded),
             callbacks=[tf.keras.callbacks.EarlyStopping('val_loss', patience=3)])

# Get the best hyperparameters and build a fresh model
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.hypermodel.build(best_hps)

print("Best hyperparameters:")
for name, value in best_hps.values.items():
    print(f"  {name}: {value}")

best_model.fit(X_train_nn_scaled, y_train_encoded,
                         epochs=50,
                         validation_data=(X_test_nn_scaled, y_test_encoded),
                         callbacks=[tf.keras.callbacks.EarlyStopping('val_loss', patience=5)],
                         verbose=0)

# Evaluate Training and Testing Accuracy
loss_train, acc_train = best_model.evaluate(X_train_nn_scaled, y_train_encoded, verbose=0)
loss_test, acc_test = best_model.evaluate(X_test_nn_scaled, y_test_encoded, verbose=0)

print(f"NN Training Accuracy: {acc_train:.4f}")
print(f"NN Test Accuracy: {acc_test:.4f}")

# Make predictions with the best model
y_pred_nn_tuned = np.argmax(best_model.predict(X_test_nn_scaled), axis=1)

# Classification Report
print("\nClassification Report for Best Tuned Neural Network:")
print(classification_report(y_test_encoded, y_pred_nn_tuned, target_names=le.classes_.astype(str)))

Reloading Tuner from keras_tuner_dir\chronotype_mlp_tuning\tuner0.json
Best hyperparameters:
  num_layers: 3
  units_0: 416
  dropout_0: 0.2
  learning_rate: 0.001
  units_1: 32
  dropout_1: 0.0
  units_2: 224
  dropout_2: 0.2
NN Training Accuracy: 0.8094
NN Test Accuracy: 0.7773
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

Classification Report for Best Tuned Neural Network:
              precision    recall  f1-score   support

         1.0       0.77      0.58      0.66       156
         2.0       0.73      0.76      0.74       344
         3.0       0.79      0.91      0.85       443
         4.0       0.78      0.70      0.74        74
         5.0       0.91      0.60      0.73       101

    accuracy                           0.78      1118
   macro avg       0.80      0.71      0.74      1118
weighted avg       0.78      0.78      0.77      1118



### Expanded Stacking Ensemble
We are now adding the Support Vector Machine (SVM) and K-Nearest Neighbors (KNN) to our stacking ensemble to potentially capture different patterns in the data and improve overall accuracy.

In [36]:
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.base import BaseEstimator, ClassifierMixin

# Wrapper class for the Keras model to make it compatible with scikit-learn stacking
class KerasClassifierWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model):
        self.model = model
        self.classes_ = np.arange(num_classes)

    def fit(self, X, y):
        # The model is already trained via Keras Tuner, so we just return self
        return self

    def predict(self, X):
        return np.argmax(self.model.predict(X, verbose=0), axis=1)

    def predict_proba(self, X):
        return self.model.predict(X, verbose=0)

# Define the base estimators including the Neural Network wrapper
estimators_with_nn = [
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=7, min_samples_split=10, class_weight='balanced', random_state=42)),
    ('xgb', XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='mlogloss')),
    ('svm', SVC(C=1, kernel='rbf', gamma='scale', probability=True, random_state=42)),
    ('nn', KerasClassifierWrapper(best_model))
]

# Initialize Stacking Classifier
stack_model_final = StackingClassifier(
    estimators=estimators_with_nn,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)

# Fit on the scaled training data
stack_model_final.fit(X_train_nn_scaled, y_train_encoded)

# Evaluation
y_train_pred_final = stack_model_final.predict(X_train_nn_scaled)
y_test_pred_final = stack_model_final.predict(X_test_nn_scaled)

stack_train_acc = accuracy_score(y_train_encoded, y_train_pred_final)
stack_test_acc = accuracy_score(y_test_encoded, y_test_pred_final)

print(f"Final Stacking Ensemble (with NN) Training Accuracy: {stack_train_acc:.4f}")
print(f"Final Stacking Ensemble (with NN) Test Accuracy: {stack_test_acc:.4f}")
print(f"Final Generalization Gap: {(stack_train_acc - stack_test_acc):.4f}")

print("\nClassification Report (Final Ensemble with NN):")
print(classification_report(y_test_encoded, y_test_pred_final, target_names=le.classes_.astype(str)))

Final Stacking Ensemble (with NN) Training Accuracy: 0.8152
Final Stacking Ensemble (with NN) Test Accuracy: 0.7728
Final Generalization Gap: 0.0424

Classification Report (Final Ensemble with NN):
              precision    recall  f1-score   support

         1.0       0.75      0.57      0.65       156
         2.0       0.72      0.77      0.75       344
         3.0       0.81      0.88      0.85       443
         4.0       0.78      0.73      0.76        74
         5.0       0.79      0.64      0.71       101

    accuracy                           0.77      1118
   macro avg       0.77      0.72      0.74      1118
weighted avg       0.77      0.77      0.77      1118



### Support Vector Machine (SVM) Classifier with Faster Hyperparameter Tuning

To address the long running time, we are re-running the SVM with a *reduced hyperparameter search space*. We will still explore both 'linear' and 'rbf' kernels but with fewer `C` (regularization) values and only the 'scale' option for `gamma` to speed up the `GridSearchCV` process.

In [37]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# Initialize SVM classifier
svm = SVC(random_state=42)

# Define a parameter grid with lower C values for stronger regularization
# This helps reduce the observed 12% overfitting gap
param_grid_svm_regularized = [
    {'C': [0.1, 1], 'kernel': ['linear']},
    {'C': [0.1, 1], 'gamma': ['scale'], 'kernel': ['rbf']}
]

# Setup GridSearchCV
grid_search_svm_reg = GridSearchCV(svm, param_grid_svm_regularized, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search_svm_reg.fit(X_train_nn_scaled, y_train_encoded)

# Best parameters
best_svm_model = grid_search_svm_reg.best_estimator_

# Predict on both sets to monitor the new generalization gap
y_train_pred_svm = best_svm_model.predict(X_train_nn_scaled)
y_test_pred_svm = best_svm_model.predict(X_test_nn_scaled)

# Scores
svm_train_acc = accuracy_score(y_train_encoded, y_train_pred_svm)
svm_test_acc = accuracy_score(y_test_encoded, y_test_pred_svm)

print(f"Best regularized parameters for SVM: {grid_search_svm_reg.best_params_}")
print(f"SVM Training Accuracy: {svm_train_acc:.4f}")
print(f"SVM Test Accuracy: {svm_test_acc:.4f}")
print(f"New SVM Generalization Gap: {(svm_train_acc - svm_test_acc):.4f}")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_encoded, y_test_pred_svm, target_names=le.classes_.astype(str)))


Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best regularized parameters for SVM: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
SVM Training Accuracy: 0.7789
SVM Test Accuracy: 0.7245
New SVM Generalization Gap: 0.0544

Classification Report (Test Set):
              precision    recall  f1-score   support

         1.0       0.64      0.54      0.59       156
         2.0       0.69      0.80      0.74       344
         3.0       0.78      0.85      0.81       443
         4.0       0.60      0.55      0.58        74
         5.0       0.97      0.33      0.49       101

    accuracy                           0.72      1118
   macro avg       0.74      0.61      0.64      1118
weighted avg       0.74      0.72      0.71      1118



# catboost



In [38]:
!pip install lightgbm catboost -q

import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# 2. CatBoost
cb_model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=0
)
cb_model.fit(X_train, y_train_encoded)
y_test_pred_cb = cb_model.predict(X_test)
cb_acc = accuracy_score(y_test_encoded, y_test_pred_cb)

print(f"CatBoost Test Accuracy: {cb_acc:.4f}")

CatBoost Test Accuracy: 0.7791
